In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

# ============================
# 1. Load and combine all frames.csv
# ============================
frames_files = glob.glob("output_*/frames.csv")  # From all scenario outputs

all_frames = pd.DataFrame()
for file in frames_files:
    df = pd.read_csv(file)
    # Extract scenario name from directory
    scenario = os.path.basename(os.path.dirname(file)).replace("output_", "")
    df['scenario'] = scenario
    all_frames = pd.concat([all_frames, df], ignore_index=True)

# ============================
# 2. Classify by connection type
# ============================
# Assuming you have naming convention like:
# wifi_baseline, ethernet_baseline, wifi_screenshare, etc.
def get_connection_type(scenario):
    scenario_lower = scenario.lower()
    if 'wifi' in scenario_lower or 'wi-fi' in scenario_lower:
        return 'Wi-Fi'
    elif 'eth' in scenario_lower or 'ethernet' in scenario_lower:
        return 'Ethernet'
    else:
        return 'Unknown'

all_frames['connection'] = all_frames['scenario'].apply(get_connection_type)

# ============================
# 3. Filter and clean jitter data
# ============================
# Convert jitter from microseconds to milliseconds
all_frames['jitter_ms'] = all_frames['jitter'] / 1000.0

# Remove outliers (optional)
jitter_data = all_frames[all_frames['jitter_ms'] < 100]  # Keep jitter < 100ms

# ============================
# 4. Create CDF plot
# ============================
plt.figure(figsize=(10, 6))

colors = {'Wi-Fi': '#FF6B6B', 'Ethernet': '#4ECDC4'}

for connection in ['Wi-Fi', 'Ethernet']:
    subset = jitter_data[jitter_data['connection'] == connection]
    if len(subset) == 0:
        continue
    
    # Sort jitter values
    sorted_jitter = np.sort(subset['jitter_ms'])
    
    # Calculate CDF
    yvals = np.arange(len(sorted_jitter)) / float(len(sorted_jitter))
    
    # Plot
    plt.plot(sorted_jitter, yvals, 
             label=f'{connection} (n={len(subset):,})',
             color=colors.get(connection, '#2C3E50'),
             linewidth=2.5)

# ============================
# 5. Add important thresholds and annotations
# ============================
# Zoom's recommended threshold (40ms)
plt.axvline(x=40, color='gray', linestyle='--', alpha=0.7, linewidth=1)
plt.text(41, 0.05, 'Zoom threshold\n(40ms)', rotation=90, fontsize=10)

# 90th percentile line (20ms)
plt.axvline(x=20, color='gray', linestyle=':', alpha=0.7, linewidth=1)
plt.text(21, 0.15, '90% below 20ms', rotation=90, fontsize=10)

# Add grid and styling
plt.grid(True, alpha=0.3, linestyle='--')
plt.xlabel('Frame-Level Jitter (ms)', fontsize=12, fontweight='bold')
plt.ylabel('Cumulative Fraction', fontsize=12, fontweight='bold')
plt.title('Cumulative Distribution of Frame-Level Jitter\nWi-Fi vs Ethernet', 
          fontsize=14, fontweight='bold', pad=20)

plt.legend(loc='lower right', fontsize=11)
plt.xlim(0, 80)  # Focus on relevant range
plt.ylim(0, 1.0)

# Add 95th percentile annotation
for connection in ['Wi-Fi', 'Ethernet']:
    subset = jitter_data[jitter_data['connection'] == connection]
    if len(subset) > 0:
        p95 = np.percentile(subset['jitter_ms'], 95)
        plt.axvline(x=p95, color=colors.get(connection), linestyle=':', alpha=0.5, linewidth=1)
        plt.text(p95+1, 0.95, f'{p95:.1f}ms', 
                color=colors.get(connection), fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# ============================
# 6. Save the plot
# ============================
plt.tight_layout()
plt.savefig('jitter_cdf.png', dpi=300, bbox_inches='tight')
plt.savefig('jitter_cdf.pdf', bbox_inches='tight')  # For LaTeX
plt.show()

# ============================
# 7. Print statistics for your paper
# ============================
print("\n=== JITTER STATISTICS FOR PAPER ===")
for connection in ['Wi-Fi', 'Ethernet']:
    subset = jitter_data[jitter_data['connection'] == connection]
    if len(subset) > 0:
        print(f"\n{connection}:")
        print(f"  Frames analyzed: {len(subset):,}")
        print(f"  Mean jitter: {subset['jitter_ms'].mean():.2f} ms")
        print(f"  Median jitter: {subset['jitter_ms'].median():.2f} ms")
        print(f"  90th percentile: {np.percentile(subset['jitter_ms'], 90):.2f} ms")
        print(f"  95th percentile: {np.percentile(subset['jitter_ms'], 95):.2f} ms")
        print(f"  99th percentile: {np.percentile(subset['jitter_ms'], 99):.2f} ms")
        print(f"  Max jitter: {subset['jitter_ms'].max():.2f} ms")
        
        # Check 90% below 20ms claim
        below_20ms = len(subset[subset['jitter_ms'] <= 20]) / len(subset) * 100
        print(f"  % frames ≤20ms: {below_20ms:.1f}%")

# Calculate Wi-Fi/Ethernet ratio at 95th percentile
wifi_95 = np.percentile(jitter_data[jitter_data['connection'] == 'Wi-Fi']['jitter_ms'], 95)
eth_95 = np.percentile(jitter_data[jitter_data['connection'] == 'Ethernet']['jitter_ms'], 95)
ratio = wifi_95 / eth_95
print(f"\nWi-Fi/Ethernet 95th percentile ratio: {ratio:.1f}x")

KeyError: 'scenario'